# Digital Cash Management Forecast: set up the project

This notebook sets up the Digital Cash Management Forecast in this Dataiku project (it also gives the project
that name), the way it runs in production. Everything is stored as CSV files. It reads the 4 curated warehouse
tables (client, monthly flows by origin, scheduled payments, monthly balances) and the load record: in the
sandbox, the datasets the notebook `warehouse_simulator.ipynb` wrote in this project; in production, the warehouse
team's CSV files in the landing folder, read-only through `EDW_LANDING` (names: `python/warehouse.py` in the
project library). It builds:

- **Raw data**: the 4 curated tables and the load record, read-only.
- **Staging**: the same tables with the standard column names.
- **Data prep**: the category mapping (a CSV dataset) applied by visual Join and Group recipes,
  obligations, model rows.
- **Training**: each training run is one model with its own **model ID**, tracked in Dataiku's
  **Experiment Tracking** and deployed as a new **version** of the saved models `forecast_ev`, `forecast_p10`,
  `forecast_p90`. A champion / challenger test activates the new version if it beats the production model.
- **Inference**: visual Score recipes with the active version, the 3-month forecast (every row says which model
  and which load produced it), the forecast archive, and the inference log (one row per run: when, which model,
  which load, how many clients).
- Two scenarios: **Nightly batch** and **Retrain, test and promote**. Each starts with the load check
  (`check_latest_load`): it stops the scenario if the latest load is incomplete, and sets `cutoff_month`.

It then trains, tests and promotes a first model, and runs the first nightly batch.

**How to use it:** in the project, import this notebook and `warehouse_simulator.ipynb` (Notebooks > New notebook
> Import). In the sandbox, run the simulator first. Then check the settings below and run all cells (Cell > Run All):
a few minutes with the small sample data. It can be run again in the same project: what exists is kept (data, the
category mapping, saved model versions, zones, scenario history), and the recipes' code is updated.

In [ ]:
PROJECT_NAME = "Digital Cash Management Forecast"
LANDING_CONNECTION = "EDW_LANDING"    # production: read-only connection to the warehouse landing folder (CSV files).
                                      # Not used when the warehouse simulator has already written the tables here.
CONNECTION = "filesystem_managed"     # where the project writes its own datasets (CSV files)
FOLDER_CONNECTION = "filesystem_folders"   # where the tracked models and the forecast archive are kept
CODE_ENV = "cashflow_forecast"        # Python code env for the recipes: an existing one (an admin creates it once)

## 1. Code environment

The recipes need CatBoost, and MLflow for Dataiku's Experiment Tracking, in an existing code env (`CODE_ENV`
above). This notebook does not create it: an admin creates it once. This checks it is there with those packages,
and otherwise stops with the request to send the admin.

In [ ]:
import dataiku

client = dataiku.api_client()
project = client.get_project(dataiku.default_project_key())
# catboost: the models; mlflow: Experiment Tracking and the saved models
PACKAGES = ("catboost>=1.2", "scikit-learn", "mlflow==3.16.0")


def check_code_env(name, packages=PACKAGES):
    """Check the code env exists and lists the packages. It is not created here: an admin creates it once."""
    request = (f"Ask a Dataiku admin to create the Python code env {name} (Python 3.10 to 3.12) with these "
               f"packages: {', '.join(packages)}. Or set CODE_ENV to an existing env that has them.")
    if name not in [e["envName"] for e in client.list_code_envs()]:
        raise RuntimeError(f"Code env {name} not found. {request}")
    try:
        listed = " ".join(client.get_code_env("PYTHON", name).get_settings().get_required_packages(as_list=True))
    except Exception:          # no right to read the env settings: it is checked when the first recipe runs
        print(f"Code env {name} found (could not read its packages)")
        return
    missing = [p for p in packages if p.split("=")[0].split(">")[0] not in listed]
    if missing:
        raise RuntimeError(f"Code env {name} lacks {', '.join(missing)}. {request}")
    print(f"Code env {name} has {', '.join(packages)}")


check_code_env(CODE_ENV)

## 2. Code

The project library gets three files: `warehouse.py` (warehouse table and column names, the load check),
`cashflow.py` (categories and the default category mapping, the inputs the models see, training and test scores)
and `tracking.py` (model IDs, Experiment Tracking, saved model versions, promotion). Each recipe
below becomes a Python recipe in the flow; the visual recipes are created in the next step.

In [ ]:
WAREHOUSE_CODE = r'''
"""Warehouse tables (project library: python/warehouse.py).

In the landing folder read through the EDW_LANDING connection (CSV files, one folder per table), the warehouse
team drops the 17 detailed tables of the data requirements (DETAIL_TABLES) and, built from them, 4 curated tables
(TABLES) plus a load record. The project reads
only the curated tables; the detailed ones stay in the warehouse for checks and drill-down.
Table names are CFF_ + the source table's name where it is known; columns follow the EDW style seen in
FCT_PAYROLL_MSTR (CSTMR_CIF, ACNT_NMBR, TOT_AMT, ..._DT, ..._FLG with Y / N). These names are our proposal until
the warehouse team confirms them: if they differ, only this file changes.
The project's staging recipe renames every column back to the standard names used by the recipes.
"""
import pandas as pd


# curated tables the project reads. standard name: (warehouse table, what the warehouse team builds it from)
TABLES = {
    "clients": ("CFF_CLIENT", "detailed tables 1, 2, 16"),
    "monthly_flows_by_origin": ("CFF_CLIENT_MNTHLY_FLOW", "detailed table 3, and CFF_SCHEDULED_PYMT"),
    "scheduled_payments": ("CFF_SCHEDULED_PYMT", "detailed tables 5, 7, 8, 9, 11, 13, 17"),
    "monthly_balances": ("CFF_CLIENT_MNTHLY_BAL", "detailed tables 2, 4"),
}
COLUMNS = {
    "clients": ["client_no", "party_type", "segment", "sub_segment", "sector", "industry", "region", "city",
                "joining_date", "legal_form", "sole_proprietor_flag", "last_annual_sales", "number_of_employees",
                "parent_client_no", "status", "closure_date", "current_accounts", "restricted_accounts"],
    # one row per client, month, direction, origin type and transaction type tag (NONE when the posting has no
    # origin or tag). Leaves out transfers between the client's own accounts and reversed postings. known_1..3: the part of the amount that came from scheduled
    # payments known 1, 2 or 3 months before the month
    "monthly_flows_by_origin": ["client_no", "month", "direction", "origin_type", "txn_type_tag", "amount", "postings",
                                "known_1", "known_2", "known_3"],
    # one row per scheduled payment, past and future; known_date: when the schedule existed
    "scheduled_payments": ["origin_type", "origin_ref", "client_no", "direction", "txn_type_tag", "due_date", "amount",
                           "known_date"],
    "monthly_balances": ["client_no", "month", "balance_end", "overdraft_limit"],
}

# the 17 detailed tables. standard name: (warehouse table, where the warehouse team takes it from)
DETAIL_TABLES = {
    "clients": ("CFF_CBG_CUSTOMER", "EDW_DM.DM_CBG_CUSTOMER"),
    "accounts": ("CFF_ACNT", "to find"),
    "transactions": ("CFF_STATEMENT_ENTRY", "EDW.FCT_STATEMENT_ENTRY"),
    "daily_balances": ("CFF_ACNT_DAILY_BAL", "to find"),
    "standing_orders": ("CFF_STANDING_ORDER", "to find"),
    "payroll_files": ("CFF_PAYROLL_MSTR", "EDW.FCT_PAYROLL_MSTR"),
    "financing_schedules": ("CFF_LOAN_FUTURE_BALANCE", "EDW.FCT_LOAN_FUTURE_BALANCE + EDW.DIM_LOAN"),
    "deposits": ("CFF_DPST", "to find"),
    "sadad_payments": ("CFF_SADAD_TRNS", "EDW.FCT_SADAD_TRNS"),
    "code_lists": ("CFF_CODE_LIST", "various"),
    "trade_finance": ("CFF_TRADE_FINANCE", "to find"),
    "card_sales": ("CFF_MRCHNT_STLMNT", "to find"),
    "corporate_cards": ("CFF_CORP_CARD", "to find"),
    "bills_issued": ("CFF_BILL_PAYMENT", "EDW.DIM_BUS_BL_ACNT + EDW.FCT_BILL_PAYMENT"),
    "client_monthly_summary": ("CFF_CBG_CUSTOMER_MNTHLY", "EDW_DM.DM_CBG_CUSTOMER"),
    "group_structure": ("CFF_CSTMR_GROUP", "to find"),
    "cheques": ("CFF_CHQ", "to find"),
}
# one row per table per load: LOAD_ID, LOAD_DTTS, AS_OF_DT, SRC_TBL_NM, TRGT_TBL_NM, ROW_CNT, LOAD_STA_CD
LOAD_RECORD = "CFF_LOAD_RECORD"
AUDIT_COLUMNS = ["ETL_LOAD_ID", "ETL_LOAD_DTTS"]      # added by the warehouse team to every table

DETAIL_COLUMNS = {
    "clients": ["client_no", "party_type", "segment", "sub_segment", "sector", "industry", "region", "city",
                "joining_date", "legal_form", "sole_proprietor_flag", "last_annual_sales", "number_of_employees",
                "parent_client_no", "status", "closure_date"],
    "accounts": ["account_no", "client_no", "product_type", "product_code", "currency", "open_date", "close_date",
                 "status", "restricted_flag", "payroll_account_flag"],
    "transactions": ["txn_id", "account_no", "posting_date", "value_date", "amount_sar", "amount_orig", "currency",
                     "exchange_rate", "debit_credit", "txn_code", "channel", "txn_type_tag", "counterparty_iban_code",
                     "counterparty_bank", "counterparty_name", "own_account_flag", "origin_type", "origin_ref",
                     "reversal_flag", "original_ref"],
    "daily_balances": ["account_no", "balance_date", "ledger_balance", "available_balance", "blocked_amount",
                       "hold_amount", "overdraft_limit"],
    "standing_orders": ["instruction_id", "account_no", "beneficiary_type", "beneficiary_iban_code", "amount",
                        "currency", "frequency", "start_date", "end_date", "next_payment_date", "status",
                        "created_date", "changed_date"],
    "payroll_files": ["FILE_ID", "DBT_CSTMR_CIF", "PAYROLL_ACNT_NMBR", "CRT_DTTS", "SLRY_TRNS_DT", "FND_DT", "TOT_AMT",
                      "NMBR_OF_EMP", "FILE_TYP_EN_NM", "FILE_SUB_TYP_EN_NM", "STA_CD", "DLT_FLG"],
    "financing_schedules": ["loan_no", "client_no", "product", "account_debited", "contract_start_date",
                            "maturity_date", "instalment_no", "due_date", "principal", "profit", "instalment_amount",
                            "paid_date", "paid_amount", "status", "early_settlement_flag", "credit_limit_total",
                            "credit_limit_used", "credit_limit_available"],
    "deposits": ["deposit_id", "client_no", "account_no", "product", "amount", "start_date", "maturity_date",
                 "expected_profit", "rollover_instruction", "status"],
    "sadad_payments": ["payment_id", "account_no", "client_no", "biller_id", "biller_name", "biller_category",
                       "bill_number", "bill_issue_date", "amount", "due_date", "paid_date", "channel"],
    "code_lists": ["code_type", "code", "meaning"],
    "trade_finance": ["instrument_id", "client_no", "instrument_type", "subtype", "amount", "currency", "issue_date",
                      "expiry_date", "payment_date", "cash_margin_account", "cash_margin_amount", "claim_amount",
                      "claim_date", "status"],
    "card_sales": ["merchant_id", "client_no", "sales_date", "settlement_date", "sales_amount", "fees", "chargebacks",
                   "net_amount_paid"],
    "corporate_cards": ["record_id", "record_type", "client_no", "statement_date", "amount_due", "due_date",
                        "paid_date", "paid_amount", "topup_date", "topup_amount"],
    "bills_issued": ["invoice_id", "client_no", "customer_iban_code", "issue_date", "due_date", "amount", "paid_date",
                     "paid_amount"],
    "client_monthly_summary": ["client_no", "month", "txn_count_in", "txn_amount_in", "txn_count_out",
                               "txn_amount_out", "online_amount", "cheque_count", "avg_balance", "payroll_salary",
                               "payroll_bonus"],
    "group_structure": ["parent_client_no", "child_client_no", "relationship_type", "start_date"],
    "cheques": ["cheque_id", "account_no", "direction", "status", "received_date", "cheque_date", "cleared_date",
                "amount", "post_dated_flag"],
}

ABBREVIATIONS = {
    "client": "CSTMR", "customer": "CSTMR", "account": "ACNT", "no": "NMBR", "number": "NMBR", "date": "DT",
    "amount": "AMT", "txn": "TRNS", "code": "CD", "flag": "FLG", "type": "TYP", "name": "NM", "status": "STA_CD",
    "currency": "CCY", "balance": "BAL", "count": "CNT", "instruction": "INSTR", "beneficiary": "BNFCRY",
    "frequency": "FREQ", "created": "CRT", "changed": "CHG", "start": "STRT", "next": "NXT", "payment": "PYMT",
    "maturity": "MTRTY", "profit": "PRFT", "principal": "PRNCPL", "instalment": "INSTL", "expected": "EXPCTD",
    "rollover": "RLOVR", "deposit": "DPST", "product": "PROD", "issue": "ISSU", "expiry": "EXPRY",
    "settlement": "STLMNT", "sales": "SLS", "fees": "FEE", "chargebacks": "CHRGBCK", "statement": "STMT",
    "record": "REC", "merchant": "MRCHNT", "invoice": "INV", "bank": "BNK", "counterparty": "CNTRPRTY",
    "origin": "ORGN", "ref": "REF", "reversal": "RVRSL", "original": "ORGNL", "channel": "CHNL", "value": "VAL",
    "posting": "PSTNG", "exchange": "EXCH", "rate": "RT", "debit": "DBT", "credit": "CRDT", "ledger": "LDGR",
    "available": "AVLBL", "blocked": "BLCKD", "hold": "HLD", "overdraft": "OVRDFT", "limit": "LMT",
    "segment": "SGMNT", "sector": "SCTR", "industry": "INDSTRY", "region": "RGN", "joining": "JOIN", "legal": "LGL",
    "proprietor": "PRPRTR", "last": "LST", "annual": "ANNL", "employees": "EMP", "closure": "CLSR", "close": "CLS",
    "open": "OPN", "restricted": "RSTRCTD", "biller": "BLLR", "category": "CTGRY", "bill": "BL", "cheque": "CHQ",
    "direction": "DRCTN", "received": "RCVD", "cleared": "CLRD", "dated": "DTD", "instrument": "INSTRMNT",
    "subtype": "SUB_TYP", "cash": "CSH", "margin": "MRGN", "claim": "CLM", "contract": "CNTRCT", "debited": "DBTD",
    "early": "ERLY", "total": "TOT", "relationship": "RLTNSHP", "month": "MNTH", "avg": "AVG", "online": "ONLN",
    "meaning": "MNNG", "parent": "PRNT", "child": "CHLD", "party": "PRTY", "sub": "SUB", "party_type": "PRTY_TYP",
    "topup": "TOPUP", "due": "DUE", "paid": "PAID", "used": "USED", "own": "OWN", "tag": "TAG", "orig": "ORIG",
    "salary": "SLRY", "bonus": "BONUS", "loan": "LOAN", "net": "NET", "current": "CRNT", "accounts": "ACNTS",
    "postings": "PSTNG_CNT", "known": "KNWN",
}


def warehouse_column(column):
    """EDW-style name of a standard column: client_no -> CSTMR_CIF, value_date -> VAL_DT, own_account_flag -> OWN_ACNT_FLG."""
    if column.isupper():                        # already a warehouse name (payroll files)
        return column
    words = column.replace("client_no", "cstmr cif").replace("_", " ").split()
    return "_".join(w.upper() if w in ("cstmr", "cif") else ABBREVIATIONS.get(w, w.upper()) for w in words)


def is_flag(column):
    """Yes / no columns: true / false in the project, Y / N in the warehouse."""
    return column.endswith("_flag")


def to_standard(df, table):
    """A curated warehouse table with the standard column names; flags Y / N become true / false."""
    out = {}
    for column in COLUMNS[table]:
        values = df[warehouse_column(column)]
        out[column] = values.astype(str).str.upper().eq("Y") if is_flag(column) else values
    return pd.DataFrame(out)


def check_latest_load():
    """The latest warehouse load must be complete before the forecast runs: every curated table loaded with status
    SUCCESS and holding the number of rows the load record says (a partial load stops here). Sets the project
    variables load_id, as_of_date and cutoff_month (the last complete month on the as-of date), which the recipes
    read. Raises an error on any problem, which stops the scenario before anything is built.
    Run by the first step of each scenario: from warehouse import check_latest_load; check_latest_load()"""
    import dataiku

    record = dataiku.Dataset(LOAD_RECORD).get_dataframe()
    latest = record.sort_values("LOAD_DTTS").iloc[-1]["LOAD_ID"]
    load = record[record["LOAD_ID"] == latest].set_index("TRGT_TBL_NM")
    problems = []
    for warehouse_table, _ in TABLES.values():
        if warehouse_table not in load.index:
            problems.append(f"{warehouse_table}: not in load {latest}")
            continue
        row = load.loc[warehouse_table]
        if row["LOAD_STA_CD"] != "SUCCESS":
            problems.append(f"{warehouse_table}: load status {row['LOAD_STA_CD']}")
        first_column = warehouse_column(COLUMNS[next(t for t, v in TABLES.items() if v[0] == warehouse_table)][0])
        count = len(dataiku.Dataset(warehouse_table).get_dataframe(columns=[first_column]))
        if count != row["ROW_CNT"]:
            problems.append(f"{warehouse_table}: {count:,} rows, the load record says {row['ROW_CNT']:,}")
    if problems:
        raise ValueError("Warehouse load not usable:\n" + "\n".join(problems))

    as_of = pd.Timestamp(load["AS_OF_DT"].max())
    as_of = (as_of.tz_convert(None) if as_of.tzinfo else as_of).normalize()
    cutoff = as_of.to_period("M") if as_of == as_of + pd.offsets.MonthEnd(0) else as_of.to_period("M") - 1
    project = dataiku.api_client().get_default_project()
    variables = project.get_variables()
    variables["standard"].update(load_id=latest, as_of_date=f"{as_of:%Y-%m-%d}", cutoff_month=str(cutoff))
    project.set_variables(variables)
    print(f"Load {latest} OK: as of {as_of:%Y-%m-%d}, cut-off month {cutoff}")
    return str(cutoff)


for _table, _columns in list(COLUMNS.items()) + list(DETAIL_COLUMNS.items()):   # names unique within a table
    _names = [warehouse_column(c) for c in _columns]
    assert len(set(_names)) == len(_names), (_table, _names)
'''

In [ ]:
LIBRARY_CODE = r'''
"""Shared settings and helpers for the cash-flow forecast recipes (project library: python/cashflow.py)."""
import numpy as np
import pandas as pd

# category code: (name, direction, predicted by the models?)
# Financing instalments and deposit maturities are fully scheduled: no model, their obligations are added as they are.
CATEGORIES = {
    "PAYROLL": ("Payroll", "OUT", True),
    "RENT": ("Rent", "OUT", True),
    "FINANCING": ("Financing instalments", "OUT", False),
    "SADAD_GOV": ("SADAD and government bills", "OUT", True),
    "SUPPLIERS": ("Supplier payments", "OUT", True),
    "CORP_CARD": ("Corporate card", "OUT", True),
    "OTHER_OUT": ("Other out", "OUT", True),
    "RECEIPTS": ("Customer receipts", "IN", True),
    "CARD_SALES": ("Card sales settlements", "IN", True),
    "CHEQUES_IN": ("Cheques in", "IN", True),
    "DEPOSIT_MAT": ("Deposit maturities", "IN", False),
    "OTHER_IN": ("Other in", "IN", True),
    "NET": ("Net cash flow", "NET", True),      # client total: money in minus money out
}
MODELLED = [c for c, (_, _, m) in CATEGORIES.items() if m]
MIN_HISTORY_MONTHS = 12     # a client needs this many months before the cut-off to get a forecast
HORIZONS = (1, 2, 3)        # months ahead
HOLDOUT_MONTHS = 6          # the last months kept aside to test the models
QUANTILES = (0.1, 0.9)      # P10 and P90
CATBOOST = {"iterations": 600, "depth": 6, "learning_rate": 0.08, "random_seed": 0, "verbose": 0}
FX_TO_SAR = {"SAR": 1.0, "USD": 3.75, "EUR": 4.05}

# Category mapping: (origin type, transaction type tag, direction) -> category, one row per combination the
# warehouse sends (NONE = no origin or tag). Anything not in the mapping is other in / other out. The setup copies
# it into the editable dataset category_mapping: edit it there (Flow > category_mapping > Edit), then run the
# scenario Nightly batch. Postings and scheduled payments use the same mapping (visual Join recipes).
DEFAULT_CATEGORY_MAPPING = [
    ("NONE", "COLLECTION", "IN", "RECEIPTS"),
    ("NONE", "SUPPLIER", "OUT", "SUPPLIERS"),
    ("NONE", "RENT", "OUT", "RENT"),
    ("NONE", "OTHER", "IN", "OTHER_IN"),
    ("NONE", "OTHER", "OUT", "OTHER_OUT"),
    ("NONE", "FEE", "OUT", "OTHER_OUT"),
    ("INVOICE", "COLLECTION", "IN", "RECEIPTS"),
    ("CARD_SETTLEMENT", "CARD_SETTLEMENT", "IN", "CARD_SALES"),
    ("CHEQUE", "CHEQUE", "IN", "CHEQUES_IN"),
    ("CORP_CARD", "CARD", "OUT", "CORP_CARD"),
    ("DEPOSIT", "OWN", "IN", "DEPOSIT_MAT"),
    ("DEPOSIT", "DEPOSIT", "IN", "DEPOSIT_MAT"),
    ("LC", "SUPPLIER", "OUT", "SUPPLIERS"),
    ("LOAN", "FINANCING", "OUT", "FINANCING"),
    ("PAYROLL", "SALARY", "OUT", "PAYROLL"),
    ("SADAD", "UTILITY", "OUT", "SADAD_GOV"),
    ("SADAD", "GOV", "OUT", "SADAD_GOV"),
    ("STANDING_ORDER", "RENT", "OUT", "RENT"),
    ("STANDING_ORDER", "SUPPLIER", "OUT", "SUPPLIERS"),
    ("STANDING_ORDER", "OTHER", "OUT", "OTHER_OUT"),
]
# formula of the join's category column: the mapped category, else other in / other out
CATEGORY_FORMULA = 'if(isBlank(mapped_category), if(direction == "IN", "OTHER_IN", "OTHER_OUT"), mapped_category)'

CAT_FEATURES = ["category", "direction", "segment", "sector", "region"]
SERIES_FEATURES = ["last_amount", "avg_3m", "same_month_last_year", "free_last", "free_avg_3m", "oblig_target_month"]
CLIENT_FEATURES = ["client_in_avg_3m", "client_out_avg_3m", "balance_end", "oblig_in_next_3m", "oblig_out_next_3m"]
OTHER_FEATURES = ["months_ahead", "target_month_of_year", "history_months", "log_scale", "series_share",
                  "log_annual_sales", "employees"]
FEATURES = CAT_FEATURES + SERIES_FEATURES + CLIENT_FEATURES + OTHER_FEATURES


def read_dataset(name):
    """A dataset as a DataFrame, with dates as plain (time zone free) timestamps."""
    import dataiku
    df = dataiku.Dataset(name).get_dataframe()
    for c in df.columns:
        if isinstance(df[c].dtype, pd.DatetimeTZDtype):
            df[c] = df[c].dt.tz_convert(None)
    return df


def month_start(dates):
    """First day of the month, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(dates)
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.to_period("M").dt.to_timestamp()


def months_between(start, end):
    """Whole months from start to end (both month timestamps)."""
    return (end.dt.year - start.dt.year) * 12 + (end.dt.month - start.dt.month)


def add_months(month, n):
    return month + pd.offsets.DateOffset(months=n)


def cutoff_month():
    """The forecast's cut-off: the last complete month of the latest checked warehouse load (project variable
    cutoff_month, set by the load check that starts each scenario)."""
    import dataiku
    return pd.Timestamp(dataiku.get_custom_variables()["cutoff_month"] + "-01")


def model_matrix(rows):
    rows = rows.assign(series_share=rows["series_scale"] / rows["scale"])
    X = rows[FEATURES].copy()
    for c in SERIES_FEATURES:
        X[c] = rows[c] / rows["series_scale"]
    for c in CLIENT_FEATURES:
        X[c] = rows[c] / rows["scale"]
    for c in CAT_FEATURES:
        X[c] = X[c].fillna("NA").astype(str)
    return X


def fit_models(rows):
    """EV model (expected value) and P10/P90 model (both quantiles in one model), on amounts / scale."""
    from catboost import CatBoostRegressor
    X, y = model_matrix(rows), rows["target"] / rows["series_scale"]
    ev = CatBoostRegressor(loss_function="RMSE", **CATBOOST)
    ev.fit(X, y, cat_features=CAT_FEATURES)
    alphas = ",".join(str(q) for q in QUANTILES)
    pq = CatBoostRegressor(loss_function=f"MultiQuantile:alpha={alphas}", **CATBOOST)
    pq.fit(X, y, cat_features=CAT_FEATURES)
    return {"ev": ev, "p10_p90": pq}


def finish(ev, p10, p90, category, index=None):
    """P10 <= EV <= P90, and no negative amounts except for the net cash flow (in SAR)."""
    ev, p10, p90 = (np.asarray(v, dtype=float) for v in (ev, p10, p90))
    p10, p90 = np.minimum(p10, ev), np.maximum(p90, ev)
    floor = np.where(np.asarray(category) == "NET", -np.inf, 0.0)
    return pd.DataFrame({"p10": np.maximum(p10, floor), "ev": np.maximum(ev, floor),
                         "p90": np.maximum(p90, floor)}, index=index).round(2)


def predict(models, rows):
    """P10, EV and P90 in SAR from the code models."""
    X = model_matrix(rows)
    scale = rows["series_scale"].values
    q = models["p10_p90"].predict(X)
    return finish(models["ev"].predict(X) * scale, q[:, 0] * scale, q[:, 1] * scale, rows["category"], rows.index)


def scores(d):
    """Test scores of predictions (columns target, p10, ev, p90, baseline): EV error (WAPE) of the model and of the
    simple guess, and the share of actual values below P10, between P10 and P90, and above P90."""
    size = d["target"].abs().sum()
    wape = lambda column: round((d["target"] - d[column]).abs().sum() / size, 4) if size else np.nan
    return pd.Series({"rows": len(d), "actual_total": round(d["target"].sum(), 0), "wape_model": wape("ev"),
                      "wape_baseline": wape("baseline") if "baseline" in d else np.nan,
                      "coverage_p10_p90": round(((d["target"] >= d["p10"]) & (d["target"] <= d["p90"])).mean(), 4),
                      "below_p10": round((d["target"] < d["p10"]).mean(), 4),
                      "above_p90": round((d["target"] > d["p90"]).mean(), 4)})


def holdout(rows):
    """The test rows: predicted month in the last 6 months before the cut-off, with the simple guess."""
    rows = rows.assign(target_month=month_start(rows["target_month"]))
    test = rows[rows["target_month"] >= add_months(cutoff_month(), -(HOLDOUT_MONTHS - 1))].copy()
    test["baseline"] = np.where(test["category"] == "NET", test["free_avg_3m"], test["free_avg_3m"].clip(lower=0))
    return test
'''

In [ ]:
TRACKING_CODE = r'''
"""Model tracking and promotion (project library: python/tracking.py).

Every training run is one model with its own model ID, recorded in Dataiku's Experiment Tracking (MLflow;
artifacts in the managed folder "experiments"): settings, cut-off, test scores on the last 6 months, the backtest
models (trained without those months, for champion / challenger tests) and three MLflow models (EV, P10, P90).
Those three are deployed as a new version (version ID = model ID) of the saved models forecast_ev, forecast_p10
and forecast_p90. The production model is the active version; promoting a model activates it in all three, and
the Score recipes always use the active version.
"""
import os
import pickle
import tempfile

import pandas as pd

EXPERIMENT = "cashflow_forecast"
FOLDER = "experiments"
SAVED_MODELS = {"ev": "forecast_ev", "p10": "forecast_p10", "p90": "forecast_p90"}   # MLflow model name: saved model
MIN_COVERAGE, MAX_COVERAGE = 0.70, 0.90     # a model is only promoted if its P10-P90 coverage is in this range


def _project():
    import dataiku
    return dataiku.api_client().get_default_project()


def _mlflow():
    import dataiku
    return _project().setup_mlflow(managed_folder=dataiku.Folder(FOLDER))


def new_model_id(cutoff):
    """CFF_<cut-off month>_<training time>, for example CFF_202610_20261104T1215 (also the saved model version)."""
    return f"CFF_{cutoff:%Y%m}_{pd.Timestamp.now():%Y%m%dT%H%M}"


def saved_model(name):
    project = _project()
    return project.get_saved_model(next(m["id"] for m in project.list_saved_models() if m["name"] == name))


def forecast_model(model, column=None):
    """An MLflow model around a CatBoost model: takes model rows, returns amounts in SAR (column: which quantile)."""
    import mlflow.pyfunc
    from cashflow import model_matrix

    class ForecastModel(mlflow.pyfunc.PythonModel):
        def __init__(self, model, column):
            self.model, self.column = model, column

        def predict(self, context, model_input, params=None):
            raw = self.model.predict(model_matrix(model_input))
            if self.column is not None:
                raw = raw[:, self.column]
            return raw * model_input["series_scale"].values

    return ForecastModel(model, column)


def log_model(model_id, params, metrics, backtest, final, reference_dataset):
    """Record one training run in Experiment Tracking, then deploy its EV, P10 and P90 models as version model_id
    of the three saved models (not activated: the promotion does that)."""
    import cloudpickle
    import cashflow
    import tracking
    cloudpickle.register_pickle_by_value(cashflow)     # the MLflow models carry their own copy of the code
    cloudpickle.register_pickle_by_value(tracking)
    with _mlflow() as mlflow:
        mlflow.set_experiment(EXPERIMENT)
        with mlflow.start_run(run_name=model_id) as run:
            mlflow.set_tag("model_id", model_id)
            mlflow.log_params({k: str(v)[:250] for k, v in params.items()})
            mlflow.log_metrics({k: float(v) for k, v in metrics.items() if pd.notna(v)})
            with tempfile.TemporaryDirectory() as tmp:
                for kind, model in backtest.items():
                    with open(os.path.join(tmp, f"backtest_{kind}.pkl"), "wb") as f:
                        pickle.dump(model, f)
                mlflow.log_artifacts(tmp, artifact_path="backtest")
            for name, model, column in (("ev", final["ev"], None), ("p10", final["p10_p90"], 0),
                                        ("p90", final["p10_p90"], 1)):
                mlflow.pyfunc.log_model(name=name, python_model=forecast_model(model, column))
            run_id = run.info.run_id
    extension = _project().get_mlflow_extension()
    code_env = _project().get_recipe("train_and_test").get_settings().get_recipe_params()["envSelection"]["envName"]
    extension.set_run_inference_info(run_id=run_id, prediction_type="REGRESSION", code_env_name=code_env,
                                     target="target")
    for name, saved_model_name in SAVED_MODELS.items():
        # evaluated on the reference sample: gives the version its input schema (the Score recipes need it)
        extension.deploy_run_model(run_id, saved_model(saved_model_name).id, version_id=model_id, model_name=name,
                                   evaluation_dataset=reference_dataset, target_column_name="target",
                                   activate=False, skip_expensive_reports=True)


def runs():
    """Every training run, newest first: model_id, run_id, trained_at, and its metrics."""
    with _mlflow() as mlflow:
        found = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["attributes.start_time DESC"])
    if found.empty:
        return pd.DataFrame(columns=["model_id", "run_id", "trained_at"])
    return found.rename(columns={"tags.model_id": "model_id", "start_time": "trained_at"})


def latest_model_id():
    """The most recently trained model."""
    found = runs()
    if found.empty:
        raise ValueError("No model trained yet: run the scenario Retrain, test and promote")
    return found.iloc[0]["model_id"]


def production_model_id():
    """The production model: the active version of the saved models, if it was ever promoted (a version Dataiku
    activated on its own when the first one was deployed does not count until the promotion)."""
    active = {name: (saved_model(sm).get_active_version() or {}).get("id") for name, sm in SAVED_MODELS.items()}
    if len(set(active.values())) != 1:
        raise ValueError(f"The saved models do not have the same active version: {active}. Activate the same model "
                         "ID in forecast_ev, forecast_p10 and forecast_p90.")
    model_id = active["ev"]
    return model_id if model_id in promoted_model_ids() else ""


def promoted_model_ids():
    """Every model ID promoted so far (read through the API: also works in a recipe that writes promotion_log)."""
    try:
        log = _project().get_dataset("promotion_log")
        columns = [c["name"] for c in log.get_schema()["columns"]]
        model, decision = columns.index("candidate_model_id"), columns.index("decision")
        return {row[model] for row in log.iter_rows() if row[decision] == "PROMOTE"}
    except Exception:            # no promotion yet
        return set()


def load_models(model_id, name="backtest"):
    """The backtest EV and P10/P90 models of a training run (for champion / challenger tests)."""
    found = runs()
    match = found[found["model_id"] == model_id]
    if match.empty:
        raise ValueError(f"Model {model_id} not found in Experiment Tracking ({EXPERIMENT})")
    with _mlflow() as mlflow:
        path = mlflow.artifacts.download_artifacts(run_id=match.iloc[0]["run_id"], artifact_path=name)
    models = {}
    for kind in ("ev", "p10_p90"):
        with open(os.path.join(path, f"{name}_{kind}.pkl"), "rb") as f:
            models[kind] = pickle.load(f)
    return models


def apply_promotion():
    """Apply the latest decision in promotion_log: on PROMOTE, activate the candidate's version in the three saved
    models. Run by a scenario step after the champion / challenger test."""
    import dataiku
    log = dataiku.Dataset("promotion_log").get_dataframe()
    last = log.sort_values("decided_at").iloc[-1]
    if last["decision"] == "PROMOTE":
        for saved_model_name in SAVED_MODELS.values():
            saved_model(saved_model_name).set_active_version(last["candidate_model_id"])
        print(f"Promoted {last['candidate_model_id']}: {last['reason']}")
    else:
        print(f"Kept the production model: {last['reason']}")
    return production_model_id()
'''

In [ ]:
RECIPES = {}

In [ ]:
RECIPES["stage_tables"] = r'''
# Staging: the 4 curated warehouse tables (CSV files in the landing folder) with the standard column names.
import dataiku

from warehouse import TABLES, to_standard

for table, (warehouse_table, _) in TABLES.items():
    raw = dataiku.Dataset(warehouse_table).get_dataframe()
    dataiku.Dataset(table).write_with_schema(to_standard(raw, table))
'''

In [ ]:
RECIPES["obligations"] = r'''
# Obligations: the scheduled payments (from the warehouse, past and future) per client, category, month due and
# month known. Their category comes from category_mapping (visual Join recipe scheduled_with_category).
import dataiku
import pandas as pd

from cashflow import add_months, cutoff_month, month_start, read_dataset

horizon_end = add_months(cutoff_month(), 4)     # needed up to 3 months after the cut-off
items = read_dataset("scheduled_with_category")
items = items[pd.to_datetime(items["due_date"]) < horizon_end]
items["due_month"] = month_start(items["due_date"])
items["known_month"] = month_start(items["known_date"])
monthly_obligations = (items.groupby(["client_no", "category", "direction", "due_month", "known_month"], as_index=False)
                       .agg(amount=("amount", "sum"), items=("origin_ref", "count")))
dataiku.Dataset("monthly_obligations").write_with_schema(monthly_obligations)
'''

In [ ]:
RECIPES["build_rows"] = r'''
# Build rows: one row per client, category, cut-off month and months ahead (1, 2, 3).
# Inputs only use what was known at the end of the cut-off month; the answer (target) is the amount in the
# predicted month minus the obligations already known at the cut-off.
# training_rows: every past cut-off with an answer. scoring_rows: the latest cut-off, to forecast.
import dataiku
import numpy as np
import pandas as pd

from cashflow import (CATEGORIES, HORIZONS, MIN_HISTORY_MONTHS, MODELLED, add_months, cutoff_month, month_start,
                      months_between, read_dataset)

last_month = cutoff_month()
flows = read_dataset("monthly_flows")
flows["month"] = month_start(flows["month"])
obl = read_dataset("monthly_obligations")
obl["due_month"], obl["known_month"] = month_start(obl["due_month"]), month_start(obl["known_month"])
clients = read_dataset("clients")

# ---- every client's months, every category, zeros where nothing moved ----------------------------
span = flows.groupby("client_no")["month"].agg(first_month="min", last_seen="max").reset_index()
span = span.merge(clients[["client_no", "status"]], on="client_no")
span["end_month"] = np.where(span["status"] == "ACTIVE", last_month, span["last_seen"])
real = [c for c in CATEGORIES if c != "NET"]
grid = pd.DataFrame([(r.client_no, m, c) for r in span.itertuples()
                     for m in pd.date_range(r.first_month, r.end_month, freq="MS") for c in real],
                    columns=["client_no", "month", "category"])
grid["direction"] = grid["category"].map({c: v[1] for c, v in CATEGORIES.items()})
values = ["amount", "known_1", "known_2", "known_3"]
grid = grid.merge(flows[["client_no", "month", "category"] + values], on=["client_no", "month", "category"], how="left")
grid[values] = grid[values].fillna(0.0)

# client totals per month, and the net cash flow as one more series
sign = np.where(grid["direction"] == "IN", 1.0, -1.0)
totals = (grid.assign(money_in=grid["amount"].where(sign > 0, 0.0), money_out=grid["amount"].where(sign < 0, 0.0))
          .groupby(["client_no", "month"], as_index=False)[["money_in", "money_out"]].sum())
net = (grid[values].mul(sign, axis=0).assign(client_no=grid["client_no"], month=grid["month"])
       .groupby(["client_no", "month"], as_index=False)[values].sum().assign(category="NET", direction="NET"))
series = pd.concat([grid, net], ignore_index=True).sort_values(["client_no", "category", "month"])

# ---- inputs at each cut-off (the month itself is the cut-off) -----------------------------------
g = series.groupby(["client_no", "category"])
roll = lambda col, n: g[col].transform(lambda s: s.rolling(n, min_periods=1).mean())
series["free"] = series["amount"] - series["known_1"]
series["last_amount"] = series["amount"]
series["avg_3m"], series["avg_12m"] = roll("amount", 3), roll("amount", 12)
series["free_last"] = series["free"]
series["free_avg_3m"] = series.groupby(["client_no", "category"])["free"].transform(
    lambda s: s.rolling(3, min_periods=1).mean())
for k in (9, 10, 11):
    series[f"lag_{k}"] = g["amount"].shift(k)

totals = totals.sort_values(["client_no", "month"])
t = totals.groupby("client_no")
for col, short in (("money_in", "in"), ("money_out", "out")):
    totals[f"client_{short}_avg_3m"] = t[col].transform(lambda s: s.rolling(3, min_periods=1).mean())
    totals[f"client_{short}_avg_12m"] = t[col].transform(lambda s: s.rolling(12, min_periods=1).mean())
totals["scale"] = ((totals["client_in_avg_12m"] + totals["client_out_avg_12m"]) / 2).clip(lower=1000.0)

bal = read_dataset("monthly_balances")
bal["month"] = month_start(bal["month"])

base = (series.merge(totals[["client_no", "month", "client_in_avg_3m", "client_out_avg_3m", "scale"]],
                     on=["client_no", "month"], how="left")
        .merge(bal, on=["client_no", "month"], how="left")
        .merge(span[["client_no", "first_month", "end_month"]], on="client_no"))
base["balance_end"] = base["balance_end"].fillna(0.0)
base["history_months"] = months_between(base["first_month"], base["month"]) + 1

# ---- obligations known at the cut-off ----------------------------------------------------------
expanded = []
for h in HORIZONS:
    o = obl.assign(cutoff=obl["due_month"].apply(lambda m: add_months(m, -h)), months_ahead=h)
    expanded.append(o[o["known_month"] <= o["cutoff"]])
expanded = pd.concat(expanded, ignore_index=True)
ob_cat = expanded.groupby(["client_no", "category", "cutoff", "months_ahead"], as_index=False)["amount"].sum()
signed = expanded.assign(amount=np.where(expanded["direction"] == "IN", 1.0, -1.0) * expanded["amount"])
ob_net = signed.groupby(["client_no", "cutoff", "months_ahead"], as_index=False)["amount"].sum().assign(category="NET")
ob_target = pd.concat([ob_cat, ob_net], ignore_index=True).rename(columns={"amount": "oblig_target_month"})
ob_next = (expanded.pivot_table(index=["client_no", "cutoff"], columns="direction", values="amount", aggfunc="sum",
                                fill_value=0.0)
           .rename(columns={"IN": "oblig_in_next_3m", "OUT": "oblig_out_next_3m"}).reset_index())
for c in ("oblig_in_next_3m", "oblig_out_next_3m"):
    if c not in ob_next:
        ob_next[c] = 0.0

# ---- one row per months ahead, with its answer -------------------------------------------------
answers = series[["client_no", "category", "month", "amount", "known_1", "known_2", "known_3"]].rename(
    columns={"month": "target_month", "amount": "actual"})
profile = clients[["client_no", "segment", "sector", "region", "last_annual_sales", "number_of_employees"]]
rows = []
for h in HORIZONS:
    r = base.assign(months_ahead=h, target_month=base["month"].apply(lambda m: add_months(m, h)))
    r["same_month_last_year"] = r[f"lag_{12 - h}"]
    r = r.merge(answers, on=["client_no", "category", "target_month"], how="left")
    r["obligations_known"] = r[f"known_{h}_y"]
    r["target"] = r["actual"] - r["obligations_known"]
    rows.append(r)
rows = pd.concat(rows, ignore_index=True).rename(columns={"month": "cutoff_month"})
rows = rows[rows["category"].isin(MODELLED)]
rows = (rows.merge(ob_target.rename(columns={"cutoff": "cutoff_month"}),
                   on=["client_no", "category", "cutoff_month", "months_ahead"], how="left")
        .merge(ob_next.rename(columns={"cutoff": "cutoff_month"}), on=["client_no", "cutoff_month"], how="left")
        .merge(profile, on="client_no", how="left"))
fill = ["oblig_target_month", "oblig_in_next_3m", "oblig_out_next_3m", "same_month_last_year"]
rows[fill] = rows[fill].fillna(0.0)
rows["target_month_of_year"] = rows["target_month"].dt.month
rows["log_scale"] = np.log10(rows["scale"])
rows["log_annual_sales"] = np.log10(rows["last_annual_sales"].clip(lower=1.0))
rows["employees"] = rows["number_of_employees"]
rows["enough_history"] = rows["history_months"] >= MIN_HISTORY_MONTHS
# a category the client did not use in the last 12 months is forecast as 0 (plus its obligations), not modelled
rows["active"] = (rows["category"] == "NET") | (rows["avg_12m"] > 0)
rows["series_scale"] = np.where(rows["category"] == "NET", rows["scale"],
                                np.maximum(rows["avg_12m"], 0.01 * rows["scale"]))

keep = ["client_no", "category", "direction", "cutoff_month", "months_ahead", "target_month", "history_months",
        "enough_history", "active", "scale", "series_scale", "segment", "sector", "region", "last_amount", "avg_3m",
        "avg_12m", "same_month_last_year", "free_last", "free_avg_3m", "client_in_avg_3m", "client_out_avg_3m",
        "balance_end", "oblig_target_month", "oblig_in_next_3m", "oblig_out_next_3m", "target_month_of_year",
        "log_scale", "log_annual_sales", "employees", "actual", "obligations_known", "target"]
rows = rows[keep]
training = rows[rows["enough_history"] & rows["active"] & rows["target"].notna()]
scoring = rows[(rows["cutoff_month"] == last_month)].drop(columns=["actual", "obligations_known", "target"])
dataiku.Dataset("training_rows").write_with_schema(training.round(4).reset_index(drop=True))
dataiku.Dataset("scoring_rows").write_with_schema(scoring.round(4).reset_index(drop=True))
'''

In [ ]:
RECIPES["train_and_test"] = r'''
# Train and test one model (one model ID, tracked in Experiment Tracking with its settings and scores):
#   backtest models: trained without the last 6 months, then tested on them (test_results)
#   final models: trained on all rows, deployed as version <model ID> of the saved models forecast_ev,
#   forecast_p10 and forecast_p90 (not active: the promotion activates it). model_reference: a sample of the
#   model's inputs (no date columns), evaluated with each version so the version knows its input columns.
import dataiku
import pandas as pd

from cashflow import (CATBOOST, CATEGORIES, FEATURES, QUANTILES, cutoff_month, fit_models, holdout, month_start,
                      predict, read_dataset, scores)
from tracking import log_model, new_model_id

cutoff = cutoff_month()
rows = read_dataset("training_rows")
rows["target_month"] = month_start(rows["target_month"])
test = holdout(rows)
backtest = fit_models(rows.drop(test.index))
test = test.join(predict(backtest, test))
final = fit_models(rows)
model_id = new_model_id(cutoff)

categories_only = test[test["category"] != "NET"]
results = pd.concat([
    test.groupby(["category", "months_ahead"]).apply(scores).reset_index(),
    test.groupby("category").apply(scores).reset_index().assign(months_ahead="ALL"),
    categories_only.groupby("months_ahead").apply(scores).reset_index().assign(category="ALL"),
    scores(categories_only).to_frame().T.assign(category="ALL", months_ahead="ALL")], ignore_index=True)
results["months_ahead"] = results["months_ahead"].astype(str)
results["category_name"] = results["category"].map({c: v[0] for c, v in CATEGORIES.items()}).fillna("All categories")
results.insert(0, "model_id", model_id)

overall = scores(categories_only)
metrics = {"wape": overall["wape_model"], "wape_baseline": overall["wape_baseline"],
           "coverage_p10_p90": overall["coverage_p10_p90"], "below_p10": overall["below_p10"],
           "above_p90": overall["above_p90"]}
for row in results[results["months_ahead"] == "ALL"].itertuples():
    if row.category != "ALL":
        metrics[f"wape_{row.category}"] = row.wape_model
params = {**CATBOOST, "quantiles": QUANTILES, "cutoff_month": f"{cutoff:%Y-%m}",
          "load_id": dataiku.get_custom_variables().get("load_id", ""), "training_rows": len(rows),
          "test_rows": len(test), "clients": rows["client_no"].nunique(), "features": ",".join(FEATURES)}
inputs = [f for f in FEATURES if f != "series_share"] + ["scale", "series_scale"]   # series_share is computed
reference = rows[inputs + ["target"]].sample(min(len(rows), 20000), random_state=0)
dataiku.Dataset("model_reference").write_with_schema(reference.reset_index(drop=True))
log_model(model_id, params, metrics, backtest, final, "model_reference")
print(f"Model {model_id}: EV error {overall['wape_model']:.1%}, coverage {overall['coverage_p10_p90']:.1%}")

dataiku.Dataset("test_results").write_with_schema(results)
'''

In [ ]:
RECIPES["champion_challenger"] = r'''
# Champion / challenger: the latest trained model (candidate) against the production model, both tested on the
# same last 6 months with their backtest models (trained without those months). If there is no production model
# yet, the candidate is promoted (the forecast needs one); its checks are noted in the reason. Otherwise it is
# promoted if
#   - its EV error is lower than the production model's (a tie keeps the production model),
#   - its P10-P90 coverage is between 70% and 90%,
#   - and it beats the simple guess (average of the last 3 months).
# The production model is the active version of the saved models. The decision is added to promotion_log; the next
# scenario step applies it (activates the candidate's version).
import dataiku
import pandas as pd

from cashflow import holdout, predict, read_dataset, scores
from tracking import MAX_COVERAGE, MIN_COVERAGE, latest_model_id, load_models, production_model_id

test = holdout(read_dataset("training_rows"))
test = test[test["category"] != "NET"]
candidate = latest_model_id()
production = production_model_id()
tested = {m: scores(test.join(predict(load_models(m, "backtest"), test)))
          for m in {candidate, production} if m}
c = tested[candidate]
p = tested.get(production) if production and production != candidate else None

checks = []
if not MIN_COVERAGE <= c["coverage_p10_p90"] <= MAX_COVERAGE:
    checks.append(f"coverage outside {MIN_COVERAGE:.0%}-{MAX_COVERAGE:.0%}")
if c["wape_model"] >= c["wape_baseline"]:
    checks.append(f"does not beat the simple guess ({c['wape_baseline']:.1%})")

if production == candidate:
    decision, reason = "KEEP", "the candidate is already the production model"
elif not production:
    decision = "PROMOTE"
    reason = (f"no production model yet, EV error {c['wape_model']:.2%}, coverage {c['coverage_p10_p90']:.1%}"
              + (f" (warning: {'; '.join(checks)})" if checks else ""))
elif not MIN_COVERAGE <= c["coverage_p10_p90"] <= MAX_COVERAGE:
    decision, reason = "KEEP", f"candidate coverage {c['coverage_p10_p90']:.1%} outside {MIN_COVERAGE:.0%}-{MAX_COVERAGE:.0%}"
elif c["wape_model"] >= c["wape_baseline"]:
    decision, reason = "KEEP", f"candidate does not beat the simple guess ({c['wape_model']:.1%} vs {c['wape_baseline']:.1%})"
elif p is not None and c["wape_model"] >= p["wape_model"]:
    decision, reason = "KEEP", f"candidate EV error {c['wape_model']:.2%} not lower than production {p['wape_model']:.2%}"
else:
    decision = "PROMOTE"
    reason = f"candidate EV error {c['wape_model']:.2%} vs production {p['wape_model']:.2%}, coverage {c['coverage_p10_p90']:.1%}"
print(decision, "-", reason)
dataiku.Dataset("promotion_log").write_with_schema(pd.DataFrame([{
    "decided_at": pd.Timestamp.now().floor("s"), "cutoff_month": dataiku.get_custom_variables()["cutoff_month"],
    "candidate_model_id": candidate, "production_model_id": production or None,
    "candidate_wape": c["wape_model"], "production_wape": p["wape_model"] if p is not None else None,
    "baseline_wape": c["wape_baseline"], "candidate_coverage": c["coverage_p10_p90"],
    "production_coverage": p["coverage_p10_p90"] if p is not None else None,
    "decision": decision, "reason": reason}]))
'''

In [ ]:
RECIPES["final_forecast"] = r'''
# Final forecast: the production model's prediction (the three Score recipes: EV, P10, P90) plus the obligations due
# that month (already scheduled, so the same amount is added to P10, EV and P90). Financing instalments and deposit
# maturities are obligations only. "NET" is the client total (money in minus money out) with its own P10, EV, P90.
# Every row says which model (model_id, the active version of the saved models), when it was promoted
# (promotion_log), and which load produced it.
import dataiku
import numpy as np
import pandas as pd

from cashflow import CATEGORIES, HORIZONS, cutoff_month, finish, month_start, read_dataset
from tracking import production_model_id

last_month = cutoff_month()
keys = ["client_no", "category", "months_ahead"]
pred = read_dataset("scored_ev").rename(columns={"prediction": "ev"})
for q in ("p10", "p90"):
    pred = pred.merge(read_dataset(f"scored_{q}")[keys + ["prediction"]].rename(columns={"prediction": q}), on=keys)
pred[["p10", "ev", "p90"]] = finish(pred["ev"], pred["p10"], pred["p90"], pred["category"], pred.index)
ok = pred["enough_history"].astype(bool) & pred["active"].astype(bool)
pred.loc[~ok, ["p10", "ev", "p90"]] = np.nan
pred.loc[pred["enough_history"].astype(bool) & ~pred["active"].astype(bool), ["p10", "ev", "p90"]] = 0.0

obl = read_dataset("monthly_obligations")
obl["due_month"], obl["known_month"] = month_start(obl["due_month"]), month_start(obl["known_month"])
clients = pred[["client_no", "history_months", "enough_history"]].drop_duplicates("client_no")
frame = clients.merge(pd.DataFrame([(c, h) for c in CATEGORIES for h in HORIZONS], columns=["category", "months_ahead"]),
                      how="cross")
frame["cutoff_month"] = last_month
frame["target_month"] = frame["months_ahead"].apply(lambda h: last_month + pd.offsets.DateOffset(months=h))
frame["direction"] = frame["category"].map({c: v[1] for c, v in CATEGORIES.items()})
frame["category_name"] = frame["category"].map({c: v[0] for c, v in CATEGORIES.items()})

known = obl[obl["known_month"] <= last_month]
by_cat = known.groupby(["client_no", "category", "due_month"], as_index=False)["amount"].sum()
net = (known.assign(amount=np.where(known["direction"] == "IN", 1.0, -1.0) * known["amount"])
       .groupby(["client_no", "due_month"], as_index=False)["amount"].sum().assign(category="NET"))
dues = pd.concat([by_cat, net], ignore_index=True).rename(columns={"due_month": "target_month", "amount": "obligations"})
frame = frame.merge(dues, on=["client_no", "category", "target_month"], how="left")
frame["obligations"] = frame["obligations"].fillna(0.0)

model = pred[keys + ["p10", "ev", "p90"]].rename(columns={"p10": "model_p10", "ev": "model_ev", "p90": "model_p90"})
frame = frame.merge(model, on=keys, how="left")
enough = frame["enough_history"].astype(bool)
for q in ("p10", "ev", "p90"):
    # no model value: a category fully scheduled, or one the client did not use in the last 12 months
    frame.loc[enough & frame[f"model_{q}"].isna(), f"model_{q}"] = 0.0
    frame[q] = frame[f"model_{q}"] + frame["obligations"]
frame.loc[~enough, ["p10", "ev", "p90", "model_p10", "model_ev", "model_p90"]] = np.nan
frame["status"] = np.where(enough, "OK", "NOT_ENOUGH_HISTORY")
frame["model_id"] = production_model_id()
log = read_dataset("promotion_log")
promoted = log[(log["decision"] == "PROMOTE") & (log["candidate_model_id"] == frame["model_id"].iloc[0])]
frame["model_promoted_at"] = promoted["decided_at"].max() if not promoted.empty else None
frame["load_id"] = dataiku.get_custom_variables().get("load_id", "")

cols = ["client_no", "cutoff_month", "months_ahead", "target_month", "category", "category_name", "direction",
        "p10", "ev", "p90", "obligations", "model_p10", "model_ev", "model_p90", "history_months", "status",
        "model_id", "model_promoted_at", "load_id"]
order = {c: i for i, c in enumerate(CATEGORIES)}
frame = frame.sort_values(["client_no", "months_ahead", "category"],
                          key=lambda s: s.map(order) if s.name == "category" else s)
dataiku.Dataset("final_forecast").write_with_schema(frame[cols].round(2).reset_index(drop=True))
'''

In [ ]:
RECIPES["archive_forecast"] = r'''
# Archive and log each forecast run:
#   forecast_archive: one CSV file per cut-off month and model (forecast_<cut-off>_<model ID>.csv)
#   inference_log: one row per run (appended): when it ran, which model and load it used, what it produced
import dataiku
import pandas as pd

from cashflow import read_dataset

forecast = read_dataset("final_forecast")
first = forecast.iloc[0]
name = f"forecast_{first['cutoff_month']:%Y-%m}_{first['model_id']}.csv"
with dataiku.Folder("forecast_archive").get_writer(name) as w:
    w.write(forecast.to_csv(index=False).encode())

variables = dataiku.get_custom_variables()
clients = forecast.drop_duplicates("client_no")
log = pd.DataFrame([{
    "run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "load_id": variables.get("load_id", ""),
    "as_of_date": variables.get("as_of_date", ""),
    "cutoff_month": f"{first['cutoff_month']:%Y-%m}",
    "model_id": first["model_id"],
    "model_promoted_at": first["model_promoted_at"],
    "clients": len(clients),
    "clients_not_enough_history": int((clients["status"] == "NOT_ENOUGH_HISTORY").sum()),
    "forecast_rows": len(forecast),
    "archive_file": name}])
dataiku.Dataset("inference_log").write_with_schema(log)
print("archived", name, "| logged run with model", first["model_id"])
'''

## 3. Build the flow

Uses the simulated warehouse tables, or links the landing files (read-only), and creates the category
mapping, the recipes (Python, Join, Group, Score) and their CSV datasets, the folders `experiments` (tracked
models) and `forecast_archive`, and the three saved models, in 5 zones. Training (folder
`experiments`), the promotion log and the saved models are only rebuilt when asked by name, so the nightly batch
never retrains by accident.

In [ ]:
import json
import time

import pandas as pd

# Safe to run again in the same project: what already exists is kept (datasets and their data, the category
# mapping, folders, saved models and their versions, zones), and the code of the Python recipes and the library
# is updated.
flow = project.get_flow()
lib = {}
for code in (LIBRARY_CODE, WAREHOUSE_CODE, TRACKING_CODE):   # the shared code is needed here too
    exec(code, lib)
RAW = [warehouse_table for warehouse_table, _ in lib["TABLES"].values()]
CSV = {"style": "excel", "charset": "utf8", "separator": ",", "quoteChar": '"', "escapeChar": "\\",
       "dateSerializationFormat": "ISO", "arrayMapFormat": "json", "parseHeaderRow": True, "compress": ""}

# Python recipes: (inputs, outputs, flow zone). Every dataset is a CSV file; experiments and forecast_archive are
# folders. The visual recipes (Join, Group, Score) are created further down.
FLOW = {
    "stage_tables": (RAW, list(lib["TABLES"]), "Staging"),
    "obligations": (["scheduled_with_category"], ["monthly_obligations"], "Data prep"),
    "build_rows": (["monthly_flows", "monthly_obligations", "clients", "monthly_balances"],
                   ["training_rows", "scoring_rows"], "Data prep"),
    "train_and_test": (["training_rows"], ["experiments", "test_results", "model_reference"], "Training"),
    "champion_challenger": (["training_rows", "experiments"], ["promotion_log"], "Training"),
    "final_forecast": (["scored_ev", "scored_p10", "scored_p90", "monthly_obligations", "promotion_log"],
                       ["final_forecast"], "Inference"),
    "archive_forecast": (["final_forecast"], ["forecast_archive", "inference_log"], "Inference"),
}
FOLDERS = ["experiments", "forecast_archive"]
APPENDED = ["promotion_log", "inference_log"]       # logs: each run adds a row
BUILT_BY_NAME = ["experiments", "promotion_log"]    # never rebuilt by building what comes after them

try:                                                # needs the project's admin right
    metadata = project.get_metadata()
    metadata["label"] = PROJECT_NAME
    project.set_metadata(metadata)
except Exception as e:
    if not any(w in str(e).lower() for w in ("unauthorized", "forbidden", "permission", "not allowed")):
        raise
    print(f"Warning: could not name the project {PROJECT_NAME!r} (no admin right on the project): name it by hand")

library = project.get_library()
python_folder = library.get_folder("python") or library.add_folder("python")
for file_name, code in (("cashflow.py", LIBRARY_CODE), ("warehouse.py", WAREHOUSE_CODE),
                        ("tracking.py", TRACKING_CODE)):
    (python_folder.get_file(file_name) or python_folder.add_file(file_name)).write(code)


# what the project already has (read once, kept up to date below)
datasets = {d["name"] for d in project.list_datasets()}
recipes = {r["name"] for r in project.list_recipes()}


def new_dataset(name):
    """Managed dataset of the project: a CSV file (comma-separated, with a header row) on CONNECTION. Kept if it
    exists."""
    if name in datasets:
        return name
    datasets.add(name)
    dataset = project.new_managed_dataset(name).with_store_into(CONNECTION).create()
    settings = dataset.get_settings()
    settings.get_raw()["formatType"] = "csv"
    settings.get_raw()["formatParams"].update(CSV)
    if name in BUILT_BY_NAME:
        settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
    settings.save()
    return name


# the curated warehouse tables and the load record: already here if the warehouse simulator wrote them (sandbox),
# otherwise CSV files in the warehouse team's landing folder, read-only (production)
simulated = [name for name in RAW + [lib["LOAD_RECORD"]] if name in datasets]
for name in RAW + [lib["LOAD_RECORD"]]:
    if name not in datasets:
        datasets.add(name)
        dataset = project.create_dataset(name, "Filesystem", params={"connection": LANDING_CONNECTION,
                                                                     "path": "/" + name},
                                          formatType="csv", formatParams=CSV)
        dataset.autodetect_settings().save()
print(f"{len(simulated)} warehouse tables already in the project, {len(RAW) + 1 - len(simulated)} linked from "
      f"{LANDING_CONNECTION}")

# category mapping: a CSV dataset with the default mapping, written only when it is created (a mapping already
# here, maybe edited, is kept). To change it: a notebook reads it, changes rows and writes it back.
if "category_mapping" not in datasets:
    dataiku.Dataset(new_dataset("category_mapping")).write_with_schema(pd.DataFrame(
        lib["DEFAULT_CATEGORY_MAPPING"],
        columns=["map_origin_type", "map_txn_type_tag", "map_direction", "mapped_category"]))
    print("category_mapping: default mapping written")

existing_folders = {f["name"]: f["id"] for f in project.list_managed_folders()}
folders = {}
for name in FOLDERS:
    if name in existing_folders:
        folders[name] = project.get_managed_folder(existing_folders[name])
        continue
    folders[name] = project.create_managed_folder(name, connection_name=FOLDER_CONNECTION)
    if name in BUILT_BY_NAME:
        folder_settings = folders[name].get_settings()
        folder_settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
        folder_settings.save()
ref = lambda name: folders[name].id if name in folders else name
existing_models = {m["name"]: m["id"] for m in project.list_saved_models()}
saved_models = {name: project.get_saved_model(existing_models[name]) if name in existing_models
                else project.create_mlflow_pyfunc_model(name, "REGRESSION") for name in lib["SAVED_MODELS"].values()}

# visual recipes: categories by Join with category_mapping (+ a formula for anything not mapped), then Group
NO_FILTER = {"enabled": False, "distinct": False}


def join_mapping(name, source, output):
    if name in recipes:
        recipe = project.get_recipe(name)
    else:
        builder = project.new_recipe("join", name)
        builder.with_input(source)
        builder.with_input("category_mapping")
        builder.with_existing_output(new_dataset(output))
        recipe = builder.create()
        recipes.add(name)
    settings = recipe.get_settings()
    if not settings.get_recipe_inputs():          # the Join creator does not always register its inputs
        settings.add_input("main", source)
        settings.add_input("main", "category_mapping")
    settings.set_payload(json.dumps({
        "virtualInputs": [{"index": i, "preFilter": NO_FILTER, "computedColumns": [],
                           "outputColumnsSelectionMode": "AUTO_NON_CONFLICTING"} for i in (0, 1)],
        "joins": [{"table1": 0, "table2": 1, "type": "LEFT", "conditionsMode": "AND", "outerJoinOnTheLeft": True,
                   "on": [{"column1": {"table": 0, "name": c}, "column2": {"table": 1, "name": "map_" + c},
                           "type": "EQ"} for c in ("origin_type", "txn_type_tag", "direction")]}],
        "selectedColumns": [], "resolvedSelectedColumns": [], "postFilter": NO_FILTER,
        "computedColumns": [{"name": "category", "type": "string", "mode": "GREL", "expr": lib["CATEGORY_FORMULA"]}],
        "enableAutoCastInJoinConditions": False}))
    settings.save()
    print("join recipe", name, "->", output)


join_mapping("join_flows_category", "monthly_flows_by_origin", "flows_with_category")
join_mapping("join_scheduled_category", "scheduled_payments", "scheduled_with_category")
if "group_monthly_flows" in recipes:
    group = project.get_recipe("group_monthly_flows").get_settings()
else:
    builder = project.new_recipe("grouping", "group_monthly_flows")
    builder.with_input("flows_with_category")
    builder.with_existing_output(new_dataset("monthly_flows"))
    group = builder.create().get_settings()
    recipes.add("group_monthly_flows")
summed = ("amount", "known_1", "known_2", "known_3", "postings")
group.set_payload(json.dumps({
    "keys": [{"column": k} for k in ("client_no", "month", "category", "direction")],
    "values": [{"column": c, "sum": True} for c in summed], "globalCount": False,
    "preFilter": NO_FILTER, "postFilter": NO_FILTER, "computedColumns": [],
    "outputColumnNameOverrides": {f"{c}_sum": c for c in summed}, "enlargeYourBits": True}))
group.save()
print("group recipe group_monthly_flows -> monthly_flows")

for name, (inputs, outputs, _) in FLOW.items():
    for item in inputs + outputs:      # inputs made by recipes created later (Score) get their dataset now
        if item not in folders:
            new_dataset(item)
    if name in recipes:                # already there: only its code and code env are updated
        recipe = project.get_recipe(name)
    else:
        builder = project.new_recipe("python", name)
        for i in inputs:
            builder.with_input(ref(i))
        for o in outputs:
            builder.with_output(ref(o), append=o in APPENDED)
        recipe = builder.create()
        recipes.add(name)
    settings = recipe.get_settings()
    settings.set_payload(RECIPES[name])
    settings.set_code_env(code_env=CODE_ENV)
    settings.save()
    print("python recipe", name, "->", ", ".join(outputs))

# Score recipes: the active version of each saved model
for short, name in lib["SAVED_MODELS"].items():
    if f"score_{short}" in recipes:
        continue
    builder = project.new_recipe("prediction_scoring", f"score_{short}")
    builder.with_input_model(saved_models[name].id)
    builder.with_input("scoring_rows")
    builder.with_existing_output(new_dataset(f"scored_{short}"))
    builder.create().rename(f"score_{short}")        # Dataiku names a Score recipe after its input
    print("score recipe", f"score_{short}", "->", f"scored_{short}")

ZONES = {"Raw data": RAW + [lib["LOAD_RECORD"]], "Staging": list(lib["TABLES"]),
         "Data prep": ["category_mapping", "flows_with_category", "monthly_flows", "scheduled_with_category"],
         "Training": [], "Inference": ["scored_ev", "scored_p10", "scored_p90"]}
for _, outputs, zone_name in FLOW.values():
    ZONES[zone_name] += [o for o in outputs if o not in ZONES[zone_name]]
# the raw data goes in Dataiku's Default zone, renamed (so there is no empty zone). It exists once another zone does.
existing_zones = {z.name: z for z in flow.list_zones()}
zones = {z: existing_zones.get(z) or flow.create_zone(z) for z in ZONES if z != "Raw data"}
zones["Raw data"] = existing_zones.get("Raw data") or flow.get_default_zone()
if zones["Raw data"].name != "Raw data":
    zone_settings = zones["Raw data"].get_settings()
    zone_settings.name = "Raw data"
    zone_settings.save()


def safe_add(zone, item):
    """Put an item in a zone (already there: nothing to do)."""
    try:
        zone.add_item(item)
    except Exception as e:
        if "already" not in str(e).lower():
            raise


for zone_name, items in ZONES.items():
    for item in items:
        safe_add(zones[zone_name], folders[item] if item in folders else project.get_dataset(item))
for model in saved_models.values():       # next to the Score recipes that use them
    safe_add(zones["Inference"], model)
print("Flow ready:", ", ".join(zones))

## 4. First run

Checks the load, trains, tests and deploys a model, runs champion / challenger (with no production model yet, the
first model is promoted, with a warning in promotion_log if its checks fail), then runs the nightly batch: score,
forecast, archive, log.

In [ ]:
def build(items, mode):
    job = project.new_job(mode).with_auto_update_schema_before_each_recipe_run(True)
    for item in items:
        if item in folders:
            job.with_output(folders[item].id, object_type="MANAGED_FOLDER")
        else:
            job.with_output(item)
    job.start_and_wait()


started = time.time()
lib["check_latest_load"]()                                   # the first step of both scenarios
build(["training_rows", "scoring_rows"], "RECURSIVE_FORCED_BUILD")
build(["experiments"], "NON_RECURSIVE_FORCED_BUILD")         # train, test, track and deploy the first model
build(["promotion_log"], "NON_RECURSIVE_FORCED_BUILD")       # champion / challenger
production = lib["apply_promotion"]()                        # activates it in the three saved models
build(["inference_log"], "RECURSIVE_FORCED_BUILD")           # score, forecast, archive, log the run
print(f"First run done in {(time.time() - started) / 60:.1f} minutes: production model {production}")

## 5. Scenarios

- **Nightly batch**: check the load, forecast with the production model, archive it and log the run.
- **Retrain, test and promote**: check the load, prepare the training data, train, test and deploy a new model,
  champion / challenger, activate it if better, then forecast with the production model.

Their triggers (daily at 06:00 and monthly on the 5th, Riyadh time) are created switched off; switch them on
in production.

In [ ]:
# Production runs. Each scenario starts with the load check (a Python step): it stops the scenario if the latest
# warehouse load is incomplete, and sets the cut-off month. The tracked models (folder experiments) and the
# promotion log are only rebuilt when built by name, so the nightly batch never retrains by accident.
def step(n, label, what, mode=None):
    common = {"id": f"step_{n}", "name": label, "enabled": True, "runConditionType": "RUN_IF_STATUS_MATCH",
              "runConditionStatuses": ["SUCCESS", "WARNING"]}
    if isinstance(what, str):           # a Python step: library function to call
        module = "warehouse" if what == "check_latest_load" else "tracking"
        return {**common, "type": "custom_python", "params": {
            "script": f"from {module} import {what}\n\n{what}()\n", "envSelection": {"envMode": "INHERIT"},
            "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}
    kind = lambda i: ("MANAGED_FOLDER", folders[i].id) if i in folders else ("DATASET", i)
    builds = [{"type": t, "itemId": item, "partitionsSpec": ""} for t, item in map(kind, what)]
    return {**common, "type": "build_flowitem", "params": {
        "builds": builds, "jobType": mode, "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}


NIGHTLY = [
    ("Check the latest warehouse load", "check_latest_load"),
    ("Forecast with the production model, archive and log it", ["inference_log"], "RECURSIVE_FORCED_BUILD"),
]
SCENARIOS = {
    "Nightly batch": ("After each warehouse load: forecast the next 3 months with the production model (the active "
                      "version of the saved models), archive it and log the run", dict(day=None), NIGHTLY),
    "Retrain, test and promote": (
        "Train and test a new model (tracked in Experiment Tracking, deployed as a saved model version), activate it if "
        "it beats the production model, then forecast with the production model", dict(day=5), [
            ("Check the latest warehouse load", "check_latest_load"),
            ("Prepare the training data", ["training_rows", "scoring_rows"], "RECURSIVE_FORCED_BUILD"),
            ("Train, test and deploy a new model", ["experiments"], "NON_RECURSIVE_FORCED_BUILD"),
            ("Champion / challenger", ["promotion_log"], "NON_RECURSIVE_FORCED_BUILD"),
            ("Promote if better", "apply_promotion"),
        ] + NIGHTLY[1:]),
}
existing_scenarios = {s["name"]: s["id"] for s in project.list_scenarios()}
for name, (description, schedule, steps) in SCENARIOS.items():
    if name in existing_scenarios:     # run again: update its steps, keep its triggers and run history
        settings = project.get_scenario(existing_scenarios[name]).get_settings()
        settings.get_raw()["description"] = description
        settings.get_raw()["params"]["steps"] = [step(n, *s) for n, s in enumerate(steps)]
        settings.save()
        print(f"Scenario updated: {name}")
        continue
    scenario = project.create_scenario(name, "step_based", {
        "active": False, "description": description,
        "params": {"steps": [step(n, *s) for n, s in enumerate(steps)]}})
    settings = scenario.get_settings()
    if schedule["day"] is None:
        settings.add_daily_trigger(hour=6, timezone="Asia/Riyadh")
    else:
        settings.add_monthly_trigger(day=schedule["day"], hour=7, timezone="Asia/Riyadh")
    settings.raw_triggers[-1]["active"] = False    # switch on in production (Automation node)
    try:
        settings.save()
        note = "trigger added, switched off"
    except Exception as e:                         # licences without time-based triggers (Free Edition)
        if "License" not in str(e):
            raise
        note = "no trigger (not in this licence): add it on the Automation node"
    print(f"Scenario ready: {name} ({note})")

## 6. Results

To use another model (for example to roll back), activate its version in the three saved models
(`forecast_ev`, `forecast_p10`, `forecast_p90`) and run the Nightly batch.

In [ ]:
print("Production model (active version of forecast_ev, forecast_p10, forecast_p90):", lib["production_model_id"]())
print("Promotion log:")
display(dataiku.Dataset("promotion_log").get_dataframe())
results = dataiku.Dataset("test_results").get_dataframe()
print("Test on the last 6 months (all months ahead together):")
display(results[results["months_ahead"].astype(str) == "ALL"][
    ["category_name", "wape_model", "wape_baseline", "coverage_p10_p90", "below_p10", "above_p90"]])
print("Inference log (one row per forecast run):")
display(dataiku.Dataset("inference_log").get_dataframe())